In [0]:
%run "../config/paths"

In [0]:
from pyspark.sql.functions import*
from pyspark.sql.window import*
from pyspark.sql.types import*

In [0]:
provider_df = spark.read.table("bronze.provider")

In [0]:
cols = [
    "prvdr_num"
    ,"fac_name"
    ,"city_name"
    ,"state_cd"
    ,"zip_cd"
    ,"fips_state_cd"
    ,"fips_cnty_cd"
    ,"cbsa_cd"
    ,"cbsa_urbn_rrl_ind"
    ,"prvdr_ctgry_cd"
    ,"prvdr_ctgry_sbtyp_cd"
    ,"gnrl_fac_type_cd"
    ,"bed_cnt"
    ,"crtfd_bed_cnt"
    ,"crtfctn_dt"
    ,"orgnl_prtcptn_dt"
    ,"pgm_trmntn_cd"
    ,"source_data"
    ,"source_code"
    ,"load_timestamp"
]
provider_silver_cols_df = provider_df.select(*cols)

In [0]:
provider_silver_cols_df = provider_silver_cols_df.na.replace(["NA", "N/A", "N"], None)

In [0]:
provider_cbsa_std_df = (
    provider_silver_cols_df.withColumn(
        "cbsa_urban_rural_ind_std",
        when(col("cbsa_urbn_rrl_ind") == "001", "U")
        .when(col("cbsa_urbn_rrl_ind") == "002", "R")
        .otherwise(col("cbsa_urbn_rrl_ind"))
    )
)

In [0]:
std_cols_df = (provider_cbsa_std_df
    .withColumn("prvdr_num", upper(col("prvdr_num"))).withColumnRenamed("prvdr_num", "provider_number")
    .withColumn("fac_name", initcap(col("fac_name"))).withColumnRenamed("fac_name", "facility_name")
    .withColumn("city_name", initcap(col("city_name")))
    .withColumn("state_cd", upper(col("state_cd"))).withColumnRenamed("state_cd", "state_code")
    .withColumn("zip_cd", upper(col("zip_cd"))).withColumnRenamed("zip_cd", "zip_code")
    .withColumn("fips_state_cd", upper(col("fips_state_cd"))).withColumnRenamed("fips_state_cd", "fips_state_code")
    .withColumn("fips_cnty_cd", upper(col("fips_cnty_cd"))).withColumnRenamed("fips_cnty_cd", "fips_county_code")
    .withColumn("cbsa_cd", upper(col("cbsa_cd"))).withColumnRenamed("cbsa_cd", "cbsa_code")
    .withColumn("cbsa_urbn_rrl_ind", upper(col("cbsa_urbn_rrl_ind"))).withColumnRenamed("cbsa_urbn_rrl_ind", "cbsa_urban_rural_ind")
    .withColumn("prvdr_ctgry_cd", upper(col("prvdr_ctgry_cd"))).withColumnRenamed("prvdr_ctgry_cd", "provider_category_code")
    .withColumn("prvdr_ctgry_sbtyp_cd", upper(col("prvdr_ctgry_sbtyp_cd"))).withColumnRenamed("prvdr_ctgry_sbtyp_cd", "provider_category_subtype_code")
    .withColumn("gnrl_fac_type_cd", upper(col("gnrl_fac_type_cd"))).withColumnRenamed("gnrl_fac_type_cd", "general_facility_type_code")
    .withColumn("bed_cnt", lower(col("bed_cnt"))).withColumnRenamed("bed_cnt", "bed_count")
    .withColumn("crtfd_bed_cnt", lower(col("crtfd_bed_cnt"))).withColumnRenamed("crtfd_bed_cnt", "certified_bed_count")
    .withColumn("crtfctn_dt", lower(col("crtfctn_dt"))).withColumnRenamed("crtfctn_dt", "certification_date")
    .withColumn("orgnl_prtcptn_dt", lower(col("orgnl_prtcptn_dt"))).withColumnRenamed("orgnl_prtcptn_dt", "original_participation_date")
    .withColumn("pgm_trmntn_cd", upper(col("pgm_trmntn_cd"))).withColumnRenamed("pgm_trmntn_cd", "program_termination_code")
)

In [0]:
##hash key

hash_df = (
        std_cols_df.withColumn
                ("provider_bk_hash", 
                        md5(concat_ws("|",
                            col("provider_number")
                    )
                )
            )
        )

In [0]:
window_spec = Window.partitionBy("provider_bk_hash").orderBy(desc("load_timestamp"))

dedup_df = (
    hash_df
        .withColumn("row_num", row_number().over(window_spec))
        .filter(col("row_num") == 1)
        .drop("row_num") # drop the col once the you get the deisred result
)

In [0]:
final_df = (
            dedup_df
            .withColumn("source_data", lit("bronze.provider"))
            .withColumn("load_timestamp", current_timestamp())

            .withColumn("certification_date", to_date(col("certification_date"),'yyyyMMdd'))
            .withColumn("original_participation_date", to_date(col("original_participation_date"),'yyyyMMdd'))

            .withColumn("bed_count", col("bed_count").cast("int"))
            .withColumn("certified_bed_count", col("certified_bed_count").cast("int"))
)

In [0]:
final_df.write.format("delta").mode("append").saveAsTable("silver.provider")